# Phase 18 — Audit of the one-standard-error model-selection procedure
**Resolves:** R2-11 (post-hoc uniform random forest; exact definition of "one standard error"; complexity ordering;
report per-outcome selected model), R2-15 (hyperparameter/selection transparency).

Reads the saved `models/ensemble_final/<tag>/strategy_comparison.csv` (notebook 07) and
 1. restates the rule exactly as coded (SE = SD of the 500 calibration-set bootstrap AUCs of the BEST-mean candidate;
    eligible = bootstrap-mean >= best-mean - SE; complexity rank single(0) < equal-weight(1) < fitted strategies(2);
    ties among singles broken by higher bootstrap-mean AUC);
 2. records, per dataset, the per-outcome selected candidate, where random forest stands, and test AUCs (with CIs) of
    the selected candidate vs random forest (paired bootstrap);
 3. repeats the rule on the larger TRAINING out-of-fold predictions (singles only) as a robustness check.
**Outputs:** `S2_selection_candidates_all.csv`, `S2_model_selection_audit.csv`, `S2_rf_vs_selected_test.csv`,
`S2_selection_oof_sensitivity.csv`

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
import ensemble as E
cand_rows, audit_rows, test_rows, oofs_rows = [], [], [], []
for tag in DATASETS:
    comp = pd.read_csv(os.path.join(ENV["MODELS"], "ensemble_final", tag, "strategy_comparison.csv"))
    winner, best_i, cutoff, elig = one_se_select(comp)
    comp = comp.sort_values("bootstrap_mean_auc", ascending=False).reset_index(drop=True)
    comp["rank_by_bootstrap_mean"] = comp.index + 1
    comp["dataset"] = tag
    comp["one_se_cutoff"] = cutoff
    comp["selected"] = comp["strategy"] == winner
    cand_rows.append(comp)
    rf = comp[comp.strategy == "single_rf"].iloc[0]; best = comp.iloc[0]
    audit_rows.append(dict(dataset=tag, best_candidate=best.strategy, best_bootstrap_mean_auc=best.bootstrap_mean_auc,
                           se_of_best=best.bootstrap_se, one_se_cutoff=cutoff, n_candidates=len(comp), n_eligible=len(elig),
                           selected_candidate=winner, selected_is_rf=(winner == "single_rf"),
                           rf_rank_of_12=int(rf.rank_by_bootstrap_mean), rf_within_1se=bool(rf.within_1se_of_best),
                           rf_gap_to_best=best.bootstrap_mean_auc - rf.bootstrap_mean_auc))
    # test performance: selected vs RF (paired)
    S = load_split(tag, ENV); y = S["pred_test"]["y_true"].values
    p_sel = strategy_proba(winner, S["oof"], S["pred_test"], seed=SEED); p_rf = S["pred_test"]["rf"].values
    a_sel = bootstrap_ci_full(y, p_sel, B=B, seed=SEED); a_rf = bootstrap_ci_full(y, p_rf, B=B, seed=SEED)
    dl = paired_bootstrap_delta(y, p_rf, p_sel, B=B, seed=SEED)
    test_rows.append(dict(dataset=tag, selected_candidate=winner, selected_test_auc=a_sel[0], selected_ci_lower=a_sel[1], selected_ci_upper=a_sel[2],
                          rf_test_auc=a_rf[0], rf_ci_lower=a_rf[1], rf_ci_upper=a_rf[2],
                          delta_rf_minus_selected=dl["delta"], delta_ci_lower=dl["ci_lower"], delta_ci_upper=dl["ci_upper"], p_two_sided=dl["p_two_sided"]))
    # robustness: rule on training OOF (single base learners only)
    oof = S["oof"]; sd = {}
    for m in MODEL_NAMES:
        mean, se = E._bootstrap_mean_se(oof["y_true"], oof[m], n_resamples=500, seed=SEED); sd[m] = (mean, se)
    bm = max(sd, key=lambda k: sd[k][0]); cut = sd[bm][0] - sd[bm][1]
    el = [m for m in MODEL_NAMES if sd[m][0] >= cut]
    for m in MODEL_NAMES:
        oofs_rows.append(dict(dataset=tag, model=m, oof_bootstrap_mean_auc=sd[m][0], oof_bootstrap_se=sd[m][1], one_se_cutoff=cut,
                              best_on_oof=(m == bm), within_1se_on_oof=m in el))
cands = pd.concat(cand_rows, ignore_index=True)
save_table(cands, ENV, "S2_selection_candidates_all.csv")
audit = pd.DataFrame(audit_rows); save_table(audit, ENV, "S2_model_selection_audit.csv")
tst = pd.DataFrame(test_rows); save_table(tst, ENV, "S2_rf_vs_selected_test.csv")
save_table(pd.DataFrame(oofs_rows), ENV, "S2_selection_oof_sensitivity.csv")
print(audit.round(4).to_string()); print(tst.round(4).to_string())

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_selection_candidates_all.csv  (48 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_model_selection_audit.csv  (4 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_rf_vs_selected_test.csv  (4 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_selection_oof_sensitivity.csv  (28 rows)
            dataset best_candidate  best_bootstrap_mean_auc  se_of_best  one_se_cutoff  n_candidates  n_eligible selected_candidate  selected_is_rf  rf_rank_of_12  rf_within_1se  rf_gap_to_best
0          pakistan     single_svm                   0.5630      0.0109         0.5521            12           5         single_svm           False              2           True          0.0011
1  arab5_depression     single_knn                   0.6692      0.0300         0.6392            12           6         single_knn           False              4           True          0.0226
2     a

**Interpretation guide for the manuscript (do not skip):**
* If `selected_is_rf` is False for a dataset, the text must NOT say random forest "was selected under this procedure".
  Say instead: random forest was within one SE of the best calibration candidate in all analyses (`rf_within_1se`) and was
  adopted as a single uniform model for the downstream analyses (SHAP, calibration, DCA, fairness); the per-outcome
  selected candidates are reported alongside (Table S-selection).
* Report `delta_rf_minus_selected` honestly. A positive, CI-excluding-zero delta means the calibration-set (n = 140-3,000)
  selection picked a model that was worse on the test set; this supports the "small calibration set is noisy" argument but must
  be stated, not hidden.